# 02 - Feature Engineering
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Lexical & structural feature extraction from URLs
2. Shannon entropy calculation
3. Typosquatting detection features
4. Punycode & IP address detection
5. Feature correlation analysis & visualization

In [ ]:
import sys
import os
import math
import re
from collections import Counter
from urllib.parse import urlparse

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load sample dataset from notebook 01
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
csv_path = os.path.join(DATA_DIR, "sample_dataset.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded dataset: {df.shape}")
else:
    print("Run notebook 01 first to generate sample_dataset.csv")
    # Fallback: create minimal sample
    df = pd.DataFrame({
        "url": [
            "https://www.google.com", "http://evil-phish.tk/login",
            "https://github.com", "http://192.168.1.1/verify.html",
        ],
        "label": [0, 1, 0, 1],
    })

df.head()

## 2.1 Feature Extraction Functions

All 24 features used by the numerical branch (MLP/CapsNet).

In [ ]:
def _shannon_entropy(text: str) -> float:
    """Calculate Shannon entropy of a string."""
    if not text:
        return 0.0
    freq = Counter(text)
    length = len(text)
    return -sum((c / length) * math.log2(c / length) for c in freq.values())


def _has_ip_address(hostname: str) -> int:
    """Check if hostname is an IP address."""
    ip_pattern = re.compile(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$")
    return 1 if ip_pattern.match(hostname) else 0


def _max_consecutive_consonants(text: str) -> int:
    """Find maximum consecutive consonants (typosquatting indicator)."""
    vowels = set("aeiouAEIOU")
    max_count = 0
    current = 0
    for char in text:
        if char.isalpha() and char not in vowels:
            current += 1
            max_count = max(max_count, current)
        else:
            current = 0
    return max_count


def _vowel_ratio(text: str) -> float:
    """Calculate ratio of vowels to total alphabetic characters."""
    alpha_chars = [c for c in text if c.isalpha()]
    if not alpha_chars:
        return 0.0
    vowels = set("aeiouAEIOU")
    return sum(1 for c in alpha_chars if c in vowels) / len(alpha_chars)


def extract_url_features(url: str) -> dict:
    """Extract all 24 numerical/lexical features from a URL."""
    parsed = urlparse(url)
    hostname = parsed.hostname or ""
    path = parsed.path or ""
    full_url = url

    features = {
        # Length-based features
        "url_length": len(full_url),
        "hostname_length": len(hostname),
        "path_length": len(path),
        # Count-based features
        "num_dots": full_url.count("."),
        "num_hyphens": full_url.count("-"),
        "num_underscores": full_url.count("_"),
        "num_slashes": full_url.count("/"),
        "num_query_params": len(parsed.query.split("&")) if parsed.query else 0,
        "num_fragments": 1 if parsed.fragment else 0,
        "num_digits": sum(c.isdigit() for c in full_url),
        "num_special_chars": sum(not c.isalnum() and c not in ".-_/:" for c in full_url),
        # Entropy
        "url_entropy": _shannon_entropy(full_url),
        "hostname_entropy": _shannon_entropy(hostname),
        # Suspicious patterns
        "has_ip_address": _has_ip_address(hostname),
        "has_punycode": 1 if hostname.startswith("xn--") else 0,
        "has_port": 1 if parsed.port and parsed.port not in (80, 443) else 0,
        "has_https": 1 if parsed.scheme == "https" else 0,
        "has_at_symbol": 1 if "@" in full_url else 0,
        "has_double_slash_redirect": 1 if "//" in path else 0,
        # Domain features
        "subdomain_count": len(hostname.split(".")) - 2 if len(hostname.split(".")) > 2 else 0,
        "tld_length": len(hostname.split(".")[-1]) if "." in hostname else 0,
        # Typosquatting indicators
        "consecutive_consonants_max": _max_consecutive_consonants(hostname),
        "vowel_ratio": _vowel_ratio(hostname),
    }

    return features


# Test on a single URL
sample = "http://paypa1-secure.com/signin/update-billing"
features = extract_url_features(sample)
print(f"URL: {sample}\n")
print(f"Features ({len(features)} total):")
for k, v in features.items():
    print(f"  {k:35s} = {v}")

## 2.2 Extract Features for Entire Dataset

In [ ]:
# Extract features for all URLs
feature_records = []
for url in df["url"]:
    feature_records.append(extract_url_features(url))

features_df = pd.DataFrame(feature_records)
features_df["label"] = df["label"].values
features_df["label_name"] = df["label"].map({0: "benign", 1: "phishing"}).values

print(f"Feature matrix shape: {features_df.shape}")
print(f"Feature names ({len(feature_records[0])}): {list(feature_records[0].keys())}")
features_df.head()

## 2.3 Feature Correlation & Analysis

In [ ]:
# Correlation heatmap
feature_cols = [c for c in features_df.columns if c not in ("label", "label_name")]
corr = features_df[feature_cols + ["label"]].corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
            xticklabels=True, yticklabels=True, linewidths=0.5)
ax.set_title("Feature Correlation Matrix (including label)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Top features correlated with label (phishing)
label_corr = corr["label"].drop("label").sort_values(ascending=False)
print("Features most correlated with phishing label:\n")
print(label_corr.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
label_corr.plot(kind="barh", ax=ax, color=label_corr.apply(lambda x: "#e74c3c" if x > 0 else "#2ecc71"))
ax.set_title("Feature Correlation with Phishing Label")
ax.set_xlabel("Pearson Correlation")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 2.4 Feature Distribution by Class

In [ ]:
# Distribution of key features by class
key_features = ["url_entropy", "hostname_entropy", "num_hyphens", "num_digits",
                "consecutive_consonants_max", "vowel_ratio"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for label, color, name in [(0, "#2ecc71", "benign"), (1, "#e74c3c", "phishing")]:
        subset = features_df[features_df["label"] == label]
        axes[i].hist(subset[feat], bins=12, alpha=0.6, label=name, color=color)
    axes[i].set_title(feat)
    axes[i].legend()

plt.suptitle("Key Feature Distributions by Class", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Save feature matrix for use in training notebooks
features_df.to_csv(os.path.join(DATA_DIR, "features_dataset.csv"), index=False)
print(f"Feature dataset saved: {features_df.shape}")
print(f"Columns: {list(features_df.columns)}")